In [1]:
# Install necessary packages
!pip install sentence_transformers pylev

# Import required libraries
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import tensorflow as tf
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import tree2conlltags
import random
import numpy as np
import warnings
import logging
import pylev

# Suppress warnings and set logging level
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore")

# Set seed for reproducibility
seed_value = 40
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load models and tokenizer
torch_device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'tuner007/pegasus_paraphrase'
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name).to(torch_device)
minilm_l12_model = SentenceTransformer('paraphrase-MiniLM-L12-v2')
mpnet_model = SentenceTransformer('paraphrase-mpnet-base-v2')


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.28GB            

...

tokenizer_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
!pip -q install sentence-transformers pylev spacy openpyxl
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:

# ============================================================
# PARC: EXPERIMENTAL SELECTION OF FILTERING THRESHOLDS
# ============================================================



import random
import warnings
import logging
import numpy as np
import pandas as pd
import torch
import pylev
import spacy

from transformers import PegasusForConditionalGeneration, PegasusTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# -------------------- 1. Reproducibility --------------------

SEED = 40
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch_device)

# -------------------- 2. Load models ------------------------

model_name = "tuner007/pegasus_paraphrase"

tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(
    model_name
).to(torch_device)
model.eval()

minilm_model = SentenceTransformer("paraphrase-MiniLM-L12-v2")
mpnet_model = SentenceTransformer("paraphrase-mpnet-base-v2")

nlp = spacy.load("en_core_web_sm")

# -------------------- 3. Experiment settings ----------------

# Use your actual representative OS subset here.
# Five sentences are only for testing that the code runs.

sentences = [
    "An operating system manages the hardware and software resources of a computer.",
    "The kernel acts as the central component of an operating system.",
    "A process is a program that is currently being executed by the processor.",
    "A thread represents an individual sequence of execution within a process.",
    "Multitasking allows an operating system to execute multiple tasks concurrently.",
    "Multiprocessing uses multiple processing units to execute computational tasks.",
    "Multithreading allows a program to perform several operations concurrently.",
    "The process control block stores information about an individual process.",
    "A process scheduler determines which process should execute next.",
    "Context switching occurs when the processor changes from one process to another.",
    "The operating system allocates processor time among competing processes.",
    "A process can move between ready, running, and waiting states.",
    "Process creation involves allocating resources and initializing execution information.",
    "Process termination releases resources allocated to the completed process.",
    "Interprocess communication enables processes to exchange information.",
    "Shared memory allows multiple processes to access a common memory region.",
    "Message passing enables processes to communicate through exchanged messages.",
    "A system call allows an application to request services from the operating system.",
    "User mode restricts applications from executing privileged processor instructions.",
    "Kernel mode provides access to privileged hardware operations and system resources.",
    "An interrupt alerts the processor that an event requires attention.",
    "An exception occurs when the processor encounters an unusual execution condition.",
    "A device driver enables the operating system to communicate with hardware.",
    "The bootloader loads the operating system kernel during system startup.",
    "The system initialization process starts essential services after the kernel loads.",
    "A monolithic kernel implements many operating system services within kernel space.",
    "A microkernel keeps essential operating system functions within a small kernel.",
    "A hybrid kernel combines architectural characteristics of monolithic and microkernels.",
    "The shell interprets user commands and requests operating system services.",
    "A command-line interface allows users to interact with the operating system through commands.",
    "A graphical user interface provides visual elements for interacting with a computer.",
    "Memory management controls the allocation and release of main memory.",
    "Virtual memory allows processes to use an address space larger than physical memory.",
    "Paging divides virtual memory into fixed-size pages.",
    "Physical memory is divided into fixed-size frames when paging is used.",
    "A page table maps virtual pages to physical memory frames.",
    "The memory management unit translates virtual addresses into physical addresses.",
    "A page fault occurs when a process accesses a page that is not currently available in memory.",
    "Demand paging loads a page into memory when it is first accessed.",
    "Page replacement algorithms select pages to remove when memory frames are needed.",
    "The least recently used algorithm replaces the page that has not been accessed for the longest time.",
    "The first-in first-out page replacement algorithm removes the oldest page in memory.",
    "Thrashing occurs when excessive paging reduces useful processor activity.",
    "Memory fragmentation reduces the usefulness of available memory regions.",
    "Internal fragmentation occurs when allocated memory contains unused space.",
    "External fragmentation occurs when free memory is divided into separated regions.",
    "Segmentation divides a program into logical units of different sizes.",
    "A memory allocator assigns memory blocks to requesting applications.",
    "Memory protection prevents one process from accessing unauthorized memory regions.",
    "A memory leak occurs when allocated memory is not properly released.",
    "The translation lookaside buffer caches recently used address translations.",
    "Swapping transfers processes or memory contents between main memory and secondary storage.",
    "Copy-on-write delays copying memory pages until a modification occurs.",
    "A heap stores dynamically allocated memory during program execution.",
    "A stack stores function call information and local variables.",
    "The operating system manages memory mappings between files and process address spaces.",
    "A real-time operating system is designed to satisfy timing constraints.",
    "Hard real-time systems must meet specified deadlines to avoid system failure.",
    "Soft real-time systems tolerate occasional deadline misses with reduced service quality.",
    "The operating system scheduler selects tasks according to its scheduling policy.",
    "First-come, first-served scheduling executes processes in arrival order.",
    "Shortest-job-first scheduling selects the process with the smallest expected execution time.",
    "Round-robin scheduling assigns each process a limited time slice.",
    "Priority scheduling selects processes according to their assigned priorities.",
    "Preemptive scheduling allows the operating system to interrupt a running process.",
    "Non-preemptive scheduling allows a process to continue until it blocks or finishes.",
    "A scheduling quantum determines how long a process may execute before being preempted.",
    "The waiting time of a process measures how long it spends in the ready queue.",
    "Turnaround time measures the interval between process submission and completion.",
    "Response time measures how quickly a system begins responding to a request.",
    "Load balancing distributes computational work across available processors.",
    "Processor affinity encourages a process to execute on a particular processor.",
    "A multicore processor contains multiple processing cores on a single chip.",
    "Symmetric multiprocessing allows processors to share access to system resources.",
    "A race condition occurs when concurrent operations produce results dependent on execution order.",
    "A critical section accesses shared data that requires coordinated execution.",
    "A mutex provides mutual exclusion when accessing a shared resource.",
    "A semaphore coordinates access to resources shared by concurrent tasks.",
    "A binary semaphore has two possible logical states.",
    "A counting semaphore tracks the availability of multiple resource instances.",
    "A monitor combines shared data with synchronization operations.",
    "A deadlock occurs when processes wait indefinitely for resources held by one another.",
    "Deadlock prevention restricts resource allocation to avoid circular waiting conditions.",
    "Deadlock avoidance evaluates resource requests before granting them.",
    "The banker's algorithm determines whether a resource allocation leaves the system in a safe state.",
    "Starvation occurs when a process waits indefinitely because other processes repeatedly receive priority.",
    "A condition variable allows threads to wait for a particular condition.",
    "Atomic operations execute indivisible updates to shared data.",
    "A spinlock repeatedly checks whether a lock has become available.",
    "A barrier synchronizes multiple threads at a designated execution point.",
    "The producer-consumer problem illustrates synchronization between tasks sharing a buffer.",
    "The readers-writers problem concerns concurrent access to shared data.",
    "A filesystem organizes files and directories on storage devices.",
    "A file is a named collection of information stored by an operating system.",
    "A directory contains references to files and other directories.",
    "File metadata stores information such as size, ownership, and modification time.",
    "File permissions determine which users may access or modify a file.",
    "An inode stores metadata and references to data blocks in many Unix filesystems.",
    "The file allocation table records information about file storage locations.",
    "Journaling filesystems record intended changes to help maintain consistency after failures.",
    "A filesystem mount makes a filesystem accessible through a directory path.",
    "A symbolic link refers to another file or directory through a pathname.",
    "A hard link provides an additional directory entry for an existing file.",
    "File locking coordinates access to files shared by multiple processes.",
    "Buffered I/O temporarily stores data in memory before transferring it.",
    "Direct memory access allows devices to transfer data without continuous processor involvement.",
    "Disk scheduling determines the order of pending storage requests.",
    "The seek time of a hard disk measures the time needed to position its read-write head.",
    "A solid-state drive stores data using nonvolatile flash memory.",
    "A disk cache keeps frequently accessed storage data in faster memory.",
    "The page cache stores file data in memory to reduce storage access operations.",
    "A virtual filesystem provides a common interface for different filesystem implementations.",
    "File descriptors identify open files or other input-output resources in Unix-like systems.",
    "A file system can enforce quotas to limit storage consumption by users.",
    "The operating system manages access to block devices and character devices.",
    "Input-output scheduling coordinates requests sent to storage and peripheral devices.",
    "An I/O buffer helps accommodate differences in data transfer rates.",
    "A device interrupt can notify the operating system when an I/O operation completes.",
    "A storage controller coordinates communication between the computer and a storage device.",
    "Disk partitioning divides a storage device into separate logical regions.",
    "A filesystem consistency check detects certain structural errors in stored filesystem data.",
    "The root directory serves as the top-level directory in a Unix-like filesystem.",
    "A file handle provides a reference to an open file or resource.",
    "The operating system can cache directory entries to accelerate pathname lookup.",
    "Virtualization allows multiple isolated computing environments to share physical hardware.",
    "A hypervisor manages virtual machines running on a physical host.",
    "A type-one hypervisor runs directly on the host hardware.",
    "A type-two hypervisor runs on top of a host operating system.",
    "A virtual machine emulates a computing environment for a guest operating system.",
    "A container isolates application processes while sharing the host kernel.",
    "Namespaces isolate selected system resources for groups of processes.",
    "Control groups regulate and monitor resource usage by groups of processes.",
    "Container images package application files and their required dependencies.",
    "The host operating system manages the physical resources used by virtual machines.",
    "A guest operating system executes inside a virtual machine.",
    "Hardware-assisted virtualization uses processor features to improve virtual machine execution.",
    "Live migration moves a running virtual machine between physical hosts.",
    "A virtual CPU represents a processor assigned to a virtual machine.",
    "Memory overcommitment allows virtual machines to request more memory than is physically available.",
    "Container orchestration systems coordinate the deployment and management of containers.",
    "A sandbox restricts the operations that an application is permitted to perform.",
    "System security protects operating system resources against unauthorized access.",
    "Authentication verifies the identity of a user or system.",
    "Authorization determines which resources an authenticated identity may access.",
    "Access control lists specify permissions associated with users and groups.",
    "The principle of least privilege limits users and processes to necessary permissions.",
    "Privilege escalation occurs when an entity gains higher permissions than intended.",
    "A security patch corrects a vulnerability or other software defect.",
    "Secure boot verifies components during the system startup process.",
    "Address space layout randomization varies memory locations to make exploitation more difficult.",
    "Data execution prevention restricts execution from designated memory regions.",
    "A firewall filters network traffic according to configured security rules.",
    "Audit logs record selected security events and system activities.",
    "Encryption protects data by transforming it into a form that requires a key to recover.",
    "A password manager stores credentials using protective security mechanisms.",
    "Multi-factor authentication requires more than one form of identity verification.",
    "A kernel vulnerability can expose privileged operating system functionality to attack.",
    "Mandatory access control enforces centrally defined security policies.",
    "Discretionary access control allows resource owners to assign certain permissions.",
    "Role-based access control assigns permissions according to organizational roles.",
    "A rootkit attempts to conceal malicious activity or maintain privileged access.",
    "Process isolation limits unintended interactions between running applications.",
    "The operating system can restrict system calls available to a process.",
    "Linux uses a monolithic kernel design with support for dynamically loadable modules.",
    "The Linux kernel manages processes, memory, devices, and system calls.",
    "A Linux distribution combines the Linux kernel with utilities and software packages.",
    "The Linux command ps displays information about running processes.",
    "The Linux command top provides a live view of process and resource activity.",
    "The kill command sends a signal to a specified process.",
    "The chmod command changes file permission bits in Unix-like systems.",
    "The chown command changes file ownership on Linux systems.",
    "The sudo utility allows authorized users to run commands with elevated privileges.",
    "The systemd service manager starts and supervises services on many Linux distributions.",
    "The Linux proc filesystem exposes process and kernel information through virtual files.",
    "The sysfs filesystem exposes information about devices and kernel objects.",
    "The Linux command mount attaches a filesystem to the directory hierarchy.",
    "The package manager installs and updates software according to distribution-specific rules.",
    "A Linux kernel module extends kernel functionality without rebuilding the entire kernel.",
    "The init process coordinates early userspace initialization on Linux systems.",
    "The Linux command free reports memory usage information.",
    "The Linux command df reports filesystem space usage.",
    "The Linux command du estimates storage space used by files and directories.",
    "The Linux command lsof lists open files and associated processes.",
    "A Linux signal notifies a process that a particular event has occurred.",
    "The Linux command nice influences the scheduling priority of a process.",
    "The Linux command renice changes the scheduling priority of an existing process.",
    "A Unix pipe transfers data between connected processes.",
    "The Linux command grep searches text for matching patterns.",
    "The Linux command find searches directory hierarchies for files.",
    "The Linux command journalctl retrieves records from the systemd journal.",
    "A Linux daemon runs in the background to provide a service.",
    "The Linux command uname displays information about the operating system and kernel.",
    "Windows uses access tokens to represent the security context of a process.",
    "The Windows Registry stores configuration information for the operating system and applications.",
    "Windows services provide background functionality independently of interactive applications.",
    "The Windows Task Manager displays running processes and resource utilization.",
    "The Windows kernel manages low-level operations and hardware interactions.",
    "The Windows NTFS filesystem supports permissions, journaling, and file metadata.",
    "The Windows Event Log records system, application, and security events.",
    "The Windows Device Manager displays installed hardware and device status.",
    "The Windows paging file provides backing storage for committed virtual memory.",
    "The Windows Service Control Manager manages registered system services.",
    "macOS uses the XNU kernel to provide core operating system functionality.",
    "The macOS APFS filesystem supports features such as snapshots and space sharing.",
    "The macOS launchd process manages system initialization and background services.",
    "The macOS Activity Monitor displays process activity and resource consumption.",
    "The macOS Gatekeeper feature helps restrict the execution of untrusted applications.",
    "The macOS Keychain stores credentials and other sensitive information.",
    "The operating system clock provides a time reference for processes and system services.",
    "Power management adjusts system behavior to balance performance and energy consumption.",
    "A suspend operation preserves system state while reducing power consumption.",
    "Hibernation saves system state to persistent storage before powering down.",
    "A watchdog timer detects certain cases of system unresponsiveness.",
    "System monitoring tracks resource usage and operating system activity.",
    "A kernel panic occurs when the kernel encounters a condition from which it cannot safely recover.",
    "System logs help administrators investigate errors and diagnose failures.",
    "A backup preserves copies of data for recovery after loss or corruption.",
    "A checkpoint records a recoverable state of a process or system.",
    "Fault tolerance allows a system to continue operating despite certain component failures.",
    "A distributed operating system coordinates resources across multiple connected computers.",
    "Network operating system services support communication and resource sharing between computers.",
    "A network socket provides an endpoint for communication between processes.",
    "The operating system manages network interfaces and their associated device drivers.",
    "A protocol stack implements the layered protocols used for network communication.",
    "A packet buffer temporarily stores network data during transmission or reception.",
    "The operating system can enforce resource limits on individual processes.",
    "Resource accounting records the processor time and memory consumed by processes.",
    "The system idle process runs when no higher-priority task is ready to execute.",
    "An operating system abstraction hides hardware-specific details from applications.",
    "The application binary interface defines conventions for interactions between compiled software components.",
    "A compatibility layer allows software designed for one environment to run in another.",
    "System configuration determines how operating system services and resources behave.",
    "The operating system validates application requests before performing privileged operations.",
    "A kernel thread performs work within kernel space rather than as a regular user process.",
    "An orphan process continues running after its original parent process terminates.",
    "A zombie process has terminated but still has an entry awaiting collection by its parent.",
    "The operating system reclaims resources when a process terminates.",
    "A file system cache improves performance by keeping frequently used data in memory."
]

# For the actual experiment, replace the list above with:
# df = pd.read_csv("/content/os_sentences.csv")
# sentences = df["sentence"].dropna().astype(str).tolist()

# Each combination will be tested against the same candidates.
LEV_THRESHOLDS = [1, 2, 5]
SIM_THRESHOLDS = [0.90, 0.95, 0.98]

# The current method retains distance > 2 and BOTH similarity
# scores > 0.95. This is represented by (2, 0.95).

NUM_CANDIDATES = 100  # Pilot value. Use 400 to match your pipeline.
MAX_LENGTH = 60

# -------------------- 4. Existing filtering helpers ---------

TIME_TERMS = {
    "today", "tomorrow", "yesterday", "now", "later", "soon",
    "week", "month", "year", "hour", "minute", "second"
}

def detect_key_words(sentence):
    doc = nlp(sentence)
    return sorted({
        token.text
        for token in doc
        if token.pos_ in {"PROPN", "NOUN", "NUM"}
        and token.text not in TIME_TERMS
    })

def count_pos(sentence, pos_tags):
    doc = nlp(sentence)
    return sum(1 for token in doc if token.pos_ in pos_tags)

def get_distance(src_txt, paraphrased_txt):
    # Preserves your existing WORD-level Levenshtein distance.
    return pylev.levenshtein(
        src_txt.lower().split(),
        paraphrased_txt.lower().split()
    )

# -------------------- 5. Generate candidates ONCE ------------

def generate_candidates(sentence, n=NUM_CANDIDATES):
    inputs = tokenizer(
        [sentence],
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(torch_device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=n,
            num_return_sequences=n,
        )

    candidates = tokenizer.batch_decode(
        outputs, skip_special_tokens=True
    )

    # Free generation tensors.
    del inputs, outputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Apply the same terminal-period convention as your code.
    candidates = [
        p.strip() + ("" if p.strip().endswith(".") else ".")
        for p in candidates
        if p.strip()
    ]

    # Deduplicate while preserving generation order.
    # Case-insensitive deduplication avoids repeated candidates.
    unique = []
    seen = set()

    for p in candidates:
        key = p.casefold()
        if key not in seen:
            seen.add(key)
            unique.append(p)

    return unique

# -------------------- 6. Compute similarity in batches --------

def batch_similarity_scores(original, candidates, encoder):
    if not candidates:
        return []

    texts = [original] + candidates

    embeddings = encoder.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Cosine similarity between original and every candidate.
    scores = np.dot(embeddings[1:], embeddings[0])
    return scores.tolist()

# -------------------- 7. Run threshold experiment ------------

all_candidate_rows = []
summary_rows = []

for sentence_index, sentence in enumerate(sentences, start=1):
    print(f"\nSentence {sentence_index}/{len(sentences)}")
    print(sentence)

    original_keywords = detect_key_words(sentence)
    original_num = count_pos(sentence, {"NUM"})

    candidates = generate_candidates(sentence)

    print("Unique generated candidates:", len(candidates))

    # Encode candidates only once per model for this sentence.
    minilm_scores = batch_similarity_scores(
        sentence, candidates, minilm_model
    )
    mpnet_scores = batch_similarity_scores(
        sentence, candidates, mpnet_model
    )

    candidate_data = []

    for candidate, minilm_score, mpnet_score in zip(
        candidates, minilm_scores, mpnet_scores
    ):
        distance = get_distance(sentence, candidate)
        candidate_keywords = detect_key_words(candidate)
        candidate_num = count_pos(candidate, {"NUM"})

        keyword_ok = all(
            word in candidate_keywords for word in original_keywords
        )
        number_ok = original_num == candidate_num

        candidate_data.append({
            "sentence_id": sentence_index,
            "original_sentence": sentence,
            "candidate": candidate,
            "word_levenshtein_distance": distance,
            "minilm_similarity": float(minilm_score),
            "mpnet_similarity": float(mpnet_score),
            "original_keywords": ", ".join(original_keywords),
            "candidate_keywords": ", ".join(candidate_keywords),
            "keyword_preservation_pass": keyword_ok,
            "number_count_preserved": number_ok,
        })

    # Evaluate every threshold combination on exactly the same outputs.
    for lev_threshold in LEV_THRESHOLDS:
        for sim_threshold in SIM_THRESHOLDS:

            retained = []

            for row in candidate_data:
                # Preserve the strict comparison in your original code:
                # distance must be > 2, and similarity must be > 0.95.
                lev_ok = (
                    row["word_levenshtein_distance"] > lev_threshold
                )
                similarity_ok = (
                    row["minilm_similarity"] > sim_threshold
                    and row["mpnet_similarity"] > sim_threshold
                )

                keep = (
                    lev_ok
                    and similarity_ok
                    and row["keyword_preservation_pass"]
                    and row["number_count_preserved"]
                )

                row_key = (
                    f"lev_gt_{lev_threshold}_sim_gt_{sim_threshold:.2f}"
                )
                row[row_key] = keep

                if keep:
                    retained.append(row["candidate"])

            summary_rows.append({
                "sentence_id": sentence_index,
                "original_sentence": sentence,
                "levenshtein_threshold": lev_threshold,
                "similarity_threshold": sim_threshold,
                "unique_generated_candidates": len(candidates),
                "retained_paraphrases": len(retained),
                "retention_rate_percent": (
                    100 * len(retained) / len(candidates)
                    if candidates else 0
                )
            })

    all_candidate_rows.extend(candidate_data)

    print("Completed threshold grid for this sentence.")

# -------------------- 8. Build result tables -----------------

summary_df = pd.DataFrame(summary_rows)
candidate_df = pd.DataFrame(all_candidate_rows)

# Overall summary across sentences.
overall_df = (
    summary_df.groupby(
        ["levenshtein_threshold", "similarity_threshold"],
        as_index=False
    )
    .agg(
        total_generated=("unique_generated_candidates", "sum"),
        total_retained=("retained_paraphrases", "sum"),
        mean_retained_per_sentence=("retained_paraphrases", "mean"),
        mean_retention_rate_percent=("retention_rate_percent", "mean")
    )
)

overall_df["overall_retention_rate_percent"] = (
    100 * overall_df["total_retained"]
    / overall_df["total_generated"].replace(0, np.nan)
)

# Include the existing configuration as an explicitly marked baseline.
overall_df["is_current_configuration"] = (
    (overall_df["levenshtein_threshold"] == 2)
    & (overall_df["similarity_threshold"] == 0.95)
)

# -------------------- 9. Create manual-review sample ----------

# Review up to 100 unique candidates that pass at least one
# configuration. The labels are intentionally left blank.
config_columns = [
    c for c in candidate_df.columns
    if c.startswith("lev_gt_")
]

if config_columns:
    candidate_df["passes_any_configuration"] = (
        candidate_df[config_columns].any(axis=1)
    )
else:
    candidate_df["passes_any_configuration"] = False

review_pool = candidate_df[
    candidate_df["passes_any_configuration"]
].drop_duplicates(["original_sentence", "candidate"])

review_sample = review_pool.sample(
    n=min(100, len(review_pool)),
    random_state=SEED
).copy() if len(review_pool) else review_pool.copy()

review_sample["meaning_preserved"] = ""
review_sample["meaningful_variation"] = ""
review_sample["key_information_preserved"] = ""
review_sample["human_valid"] = ""
review_sample["reviewer_comments"] = ""

# -------------------- 10. Export to Excel ---------------------

output_file = "/content/parc_threshold_experiment.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    overall_df.to_excel(
        writer, sheet_name="Overall_Summary", index=False
    )
    summary_df.to_excel(
        writer, sheet_name="Per_Sentence_Results", index=False
    )
    candidate_df.to_excel(
        writer, sheet_name="Candidate_Audit", index=False
    )
    review_sample.to_excel(
        writer, sheet_name="Manual_Review", index=False
    )

print("\nExperiment complete.")
print("\nOverall results:")
print(overall_df.to_string(index=False))
print("\nSaved to:", output_file)

try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print("Download the Excel file from the notebook's file browser.")


Device: cuda


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Sentence 1/238
An operating system manages the hardware and software resources of a computer.
Unique generated candidates: 95
Completed threshold grid for this sentence.

Sentence 2/238
The kernel acts as the central component of an operating system.
Unique generated candidates: 89
Completed threshold grid for this sentence.

Sentence 3/238
A process is a program that is currently being executed by the processor.
Unique generated candidates: 90
Completed threshold grid for this sentence.

Sentence 4/238
A thread represents an individual sequence of execution within a process.
Unique generated candidates: 95
Completed threshold grid for this sentence.

Sentence 5/238
Multitasking allows an operating system to execute multiple tasks concurrently.
Unique generated candidates: 98
Completed threshold grid for this sentence.

Sentence 6/238
Multiprocessing uses multiple processing units to execute computational tasks.
Unique generated candidates: 94
Completed threshold grid for this sentenc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>